# Nhận diện biển số bãi xe với YOLO26

Notebook này fine-tune **Ultralytics YOLO26 Detect** để tìm `car`, `motorcycle`, `plate`. Model trả về vị trí biển số, **không đọc chuỗi ký tự**. Muốn có kết quả như `51K-029.30`, hệ thống cần thêm: crop biển → chỉnh phối cảnh → OCR → kiểm tra cú pháp.

Pipeline: audit dữ liệu → chuẩn hóa nhãn → train → validation/test → đo latency → export. Dataset hiện có 2.033 ảnh (1.423/407/203) và 4.726 object. Audit ban đầu phát hiện 62 polygon trộn với box; notebook đổi polygon thành bounding box trên **bản sao**, không sửa dữ liệu gốc.

## 1. Runtime và cấu hình

Trên Colab chọn **Runtime → Change runtime type → T4 GPU**. `yolo26n` là baseline nhanh; chỉ đổi sang `yolo26s` nếu recall biển nhỏ chưa đủ. 640 là baseline phù hợp dữ liệu hiện tại; 960 có số pixel gấp 2,25 lần nên phải benchmark trước khi dùng.

In [ ]:
%pip install -q -U ultralytics pyyaml

import platform, random, re, shutil, statistics, time, zipfile
from collections import Counter
from pathlib import Path
import cv2, matplotlib.pyplot as plt, numpy as np, torch, ultralytics, yaml
from ultralytics import YOLO

IN_COLAB = 'google.colab' in str(get_ipython())
HAS_CUDA = torch.cuda.is_available()
DEVICE = 0 if HAS_CUDA else 'cpu'
print('Ultralytics:', ultralytics.__version__, '| PyTorch:', torch.__version__)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not HAS_CUDA:
    print('\n⚠️ Không phát hiện NVIDIA GPU. Notebook vẫn chạy bằng CPU nhưng train sẽ rất chậm.')
    print('Nếu dùng Colab: Runtime → Change runtime type → Hardware accelerator → T4 GPU,')
    print('sau đó Save và chạy lại từ đầu. Có thể kiểm tra thêm bằng lệnh !nvidia-smi.')

# Notebook không tự mount Drive để tránh popup cấp quyền khi chạy qua VS Code.
# Cách 1: chuột phải folder dataset trong VS Code → Upload to Colab.
# Cách 2: Ctrl+Shift+P → Colab: Mount Google Drive to Server... (chỉ làm một lần/server).
dataset_candidates = [
    Path('/content/Vietnam License plate.v1i.yolo26'),
    Path('/content/Vietnam License plate.v1i.yolo26.zip'),
    Path('/content/drive/MyDrive/Yolo_Parking/Vietnam License plate.v1i.yolo26'),
    Path.cwd() / 'Vietnam License plate.v1i.yolo26',
]
DATASET_INPUT = next((p for p in dataset_candidates if p.exists()), dataset_candidates[0])
MODEL_NAME, IMG_SIZE, EPOCHS, PATIENCE, SEED = 'yolo26n.pt', 640, 100, 25, 42
BATCH = 0.70 if HAS_CUDA else 4  # auto VRAM chỉ dùng cho train; CPU dùng batch nhỏ
VAL_BATCH = (16 if IMG_SIZE <= 640 else 8) if HAS_CUDA else 4
WORK_DIR = Path('/content/parking_yolo26') if IN_COLAB else Path.cwd()/'parking_yolo26_artifacts'
DATA_DIR, RUNS_DIR = WORK_DIR/'dataset', WORK_DIR/'runs'
RUN_NAME = f'vn_plate_{Path(MODEL_NAME).stem}_{IMG_SIZE}'
WORK_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print({'dataset': str(DATASET_INPUT), 'model': MODEL_NAME, 'imgsz': IMG_SIZE, 'batch': BATCH})

## 2. Copy, audit và chuẩn hóa nhãn

YOLO Detect chuẩn dùng `class x_center y_center width height`, tọa độ trong `[0,1]`. Cell kiểm tra ảnh/label bị thiếu, class id, box lỗi và data leakage theo tên ảnh gốc. Polygon `class x1 y1 ...` được đổi thành box bao ngoài. Nếu cell dừng, hãy sửa dữ liệu thay vì bỏ qua lỗi.

In [ ]:
required = ['DATASET_INPUT', 'WORK_DIR', 'DATA_DIR']
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError('Runtime vừa khởi động lại hoặc bạn chưa chạy cell 1. Hãy chọn Runtime → Run all (Chạy tất cả), hoặc chạy lại cell Runtime và cấu hình trước. Thiếu biến: ' + ', '.join(missing))

def find_root(source):
    source = source.expanduser().resolve()
    if not source.exists():
        checked = '\n'.join(f'  - {p}' for p in dataset_candidates)
        raise FileNotFoundError('Không tìm thấy dataset. Đã kiểm tra:\n' + checked + '\nTrong VS Code: chuột phải folder dataset → Upload to Colab, rồi chạy lại từ cell 1.')
    if source.is_file() and source.suffix.lower() == '.zip':
        out = WORK_DIR/'unzipped'
        if out.exists(): shutil.rmtree(out)
        out.mkdir(parents=True)
        with zipfile.ZipFile(source) as z: z.extractall(out)
        files = list(out.rglob('data.yaml'))
        if len(files) != 1: raise RuntimeError(f'Cần đúng 1 data.yaml, thấy {len(files)}')
        return files[0].parent
    if source.is_dir() and (source/'data.yaml').exists(): return source
    raise RuntimeError('DATASET_INPUT phải là folder chứa data.yaml hoặc file zip')

source_root = find_root(DATASET_INPUT)
if DATA_DIR.exists(): shutil.rmtree(DATA_DIR)
shutil.copytree(source_root, DATA_DIR)
with open(DATA_DIR/'data.yaml', encoding='utf-8') as f: raw_cfg = yaml.safe_load(f)
NAMES = raw_cfg['names']
NAMES = dict(enumerate(NAMES)) if isinstance(NAMES, list) else {int(k):v for k,v in NAMES.items()}
stats, counts, errors = Counter(), Counter(), []

def to_box(line, path, n):
    p = line.split()
    if not p: return None
    try: cls, v = int(float(p[0])), [float(x) for x in p[1:]]
    except ValueError: raise ValueError(f'{path}:{n}: giá trị không phải số')
    if cls not in NAMES: raise ValueError(f'{path}:{n}: class {cls} không tồn tại')
    if len(v) == 4:
        x,y,w,h = v; stats['box'] += 1
    elif len(v) >= 6 and len(v)%2 == 0:
        pts = np.array(v).reshape(-1,2); x1,y1 = pts.min(0); x2,y2 = pts.max(0)
        x,y,w,h = (x1+x2)/2,(y1+y2)/2,x2-x1,y2-y1; stats['polygon_to_box'] += 1
    else: raise ValueError(f'{path}:{n}: sai số cột')
    a = np.array([x,y,w,h])
    if not np.isfinite(a).all() or (a<0).any() or (a>1).any() or w<=0 or h<=0:
        raise ValueError(f'{path}:{n}: box lỗi {a.tolist()}')
    counts[cls] += 1
    return f'{cls} {x:.8f} {y:.8f} {w:.8f} {h:.8f}'

for split in ('train','valid','test'):
    for f in (DATA_DIR/split/'labels').glob('*.txt'):
        rows=[]
        for n,line in enumerate(f.read_text(encoding='utf-8').splitlines(),1):
            try:
                row=to_box(line,f,n)
                if row: rows.append(row)
            except ValueError as e: errors.append(str(e))
        f.write_text('\n'.join(rows)+('\n' if rows else ''),encoding='utf-8')
if errors: raise RuntimeError('Lỗi nhãn:\n'+'\n'.join(errors[:20]))
print('Format:',dict(stats),'| Objects:',{NAMES[k]:counts[k] for k in NAMES})

In [ ]:
EXT={'.jpg','.jpeg','.png','.bmp','.webp'}; keys={}; summary={}
def original_key(stem): return re.sub(r'_(jpg|jpeg|png)\.rf\.[^.]+$','',stem,flags=re.I)
for split in ('train','valid','test'):
    images=[p for p in (DATA_DIR/split/'images').iterdir() if p.suffix.lower() in EXT]
    labels=list((DATA_DIR/split/'labels').glob('*.txt'))
    ims,lbs={p.stem for p in images},{p.stem for p in labels}
    if ims-lbs or lbs-ims: raise RuntimeError(f'{split}: thiếu {len(ims-lbs)} label, {len(lbs-ims)} image')
    keys[split]={original_key(p.stem) for p in images}; summary[split]=len(images)
for a,b in (('train','valid'),('train','test'),('valid','test')):
    overlap=keys[a]&keys[b]
    if overlap: raise RuntimeError(f'Data leakage {a}/{b}: {list(overlap)[:5]}')
DATA_YAML=WORK_DIR/'data_prepared.yaml'
cfg={'path':str(DATA_DIR.resolve()),'train':'train/images','val':'valid/images','test':'test/images','names':NAMES}
DATA_YAML.write_text(yaml.safe_dump(cfg,sort_keys=False,allow_unicode=True),encoding='utf-8')
print('Audit OK:',summary); print(DATA_YAML.read_text())

### Kiểm tra nhãn bằng mắt

Code biết format đúng nhưng không biết box có ôm đúng vật thể. Hãy xem 20–50 ảnh, nhất là ảnh tối, biển nhỏ, xe máy, che khuất và góc xiên. Nhãn sai nhất quán thường hại hơn thiếu vài ảnh.

In [ ]:
def draw_labels(path):
    im=cv2.imread(str(path)); h,w=im.shape[:2]; label=path.parent.parent/'labels'/f'{path.stem}.txt'
    for row in label.read_text().splitlines():
        c,x,y,bw,bh=map(float,row.split()); x1,y1=int((x-bw/2)*w),int((y-bh/2)*h); x2,y2=int((x+bw/2)*w),int((y+bh/2)*h)
        color=[(65,105,225),(255,140,0),(50,205,50)][int(c)%3]
        cv2.rectangle(im,(x1,y1),(x2,y2),color,2); cv2.putText(im,NAMES[int(c)],(x1,max(18,y1-5)),0,.55,color,2)
    return cv2.cvtColor(im,cv2.COLOR_BGR2RGB)
train_images=[p for p in (DATA_DIR/'train/images').iterdir() if p.suffix.lower() in EXT]
fig,axes=plt.subplots(3,3,figsize=(15,15))
for ax,p in zip(axes.flat,random.sample(train_images,9)): ax.imshow(draw_labels(p)); ax.axis('off')
plt.tight_layout()

## 3. Fine-tune YOLO26

Dùng pretrained weights thay vì train từ đầu. AMP giảm VRAM; cache disk tránh đầy RAM. Augmentation vừa phải và không lật ngang vì biển phản chiếu là dữ liệu giả. `close_mosaic=10` trả về ảnh tự nhiên ở 10 epoch cuối. Nếu Colab ngắt, chạy cell resume; model bàn giao là `best.pt`, không phải `last.pt`.

In [ ]:
model=YOLO(MODEL_NAME)
model.train(data=str(DATA_YAML),epochs=EPOCHS,patience=PATIENCE,imgsz=IMG_SIZE,batch=BATCH,
    device=DEVICE,workers=2,cache='disk',amp=True,optimizer='auto',cos_lr=True,close_mosaic=10,
    degrees=3,translate=.08,scale=.35,shear=1,perspective=.0003,fliplr=0,flipud=0,mosaic=.5,mixup=0,
    seed=SEED,deterministic=True,plots=True,save_period=10,project=str(RUNS_DIR),name=RUN_NAME,exist_ok=True)
BEST_PT=RUNS_DIR/RUN_NAME/'weights/best.pt'
assert BEST_PT.exists(); print('Best:',BEST_PT)

# Resume nếu phiên trước bị ngắt (bỏ dấu # rồi chỉ chạy 2 dòng dưới):
# last=YOLO(str(RUNS_DIR/RUN_NAME/'weights/last.pt'))
# last.train(resume=True)

## 4. Đánh giá

Precision cao nghĩa là ít báo nhầm; recall cao nghĩa là ít bỏ sót. mAP50 dễ hơn, còn mAP50–95 phản ánh độ khít box—crop cắt mất ký tự sẽ làm OCR tệ. Chọn model/threshold bằng validation và xem riêng class `plate`; chỉ chạy test một lần sau khi chốt, nếu không sẽ vô tình tune vào test.

In [ ]:
best=YOLO(str(BEST_PT))
val=best.val(data=str(DATA_YAML),split='val',imgsz=IMG_SIZE,batch=VAL_BATCH,device=DEVICE,plots=True,nms=True)
print('Val mAP50:',float(val.box.map50),'| mAP50-95:',float(val.box.map))
print('Per class:',dict(zip(NAMES.values(),map(float,val.box.maps))))

# CHỈ chạy sau khi đã chốt bằng validation:
# test=best.val(data=str(DATA_YAML),split='test',imgsz=IMG_SIZE,batch=VAL_BATCH,device=DEVICE,plots=True,nms=True)
# print('FINAL test:',float(test.box.map50),float(test.box.map))

In [ ]:
test_images=[p for p in (DATA_DIR/'test/images').iterdir() if p.suffix.lower() in EXT]
samples=random.sample(test_images,6)
preds=best.predict(samples,imgsz=IMG_SIZE,conf=.25,iou=.70,device=DEVICE,nms=True,verbose=False)
fig,axes=plt.subplots(2,3,figsize=(16,10))
for ax,r in zip(axes.flat,preds): ax.imshow(cv2.cvtColor(r.plot(),cv2.COLOR_BGR2RGB)); ax.axis('off')
plt.tight_layout()

# Crop class plate để đưa sang OCR. conf=.25 chỉ là điểm khởi đầu, phải tune trên validation.
plate_id=next(k for k,v in NAMES.items() if v.lower()=='plate'); crops=[]
for r in preds:
    for box in ([] if r.boxes is None else r.boxes):
        if int(box.cls.item())!=plate_id: continue
        x1,y1,x2,y2=box.xyxy[0].cpu().round().int().tolist(); pad=4
        x1,y1=max(0,x1-pad),max(0,y1-pad); x2,y2=min(r.orig_img.shape[1],x2+pad),min(r.orig_img.shape[0],y2+pad)
        if x2>x1 and y2>y1: crops.append((r.orig_img[y1:y2,x1:x2],float(box.conf.item())))
if crops:
    fig,axes=plt.subplots(1,min(8,len(crops)),figsize=(18,4)); axes=np.atleast_1d(axes)
    for ax,(im,c) in zip(axes,crops[:8]): ax.imshow(cv2.cvtColor(im,cv2.COLOR_BGR2RGB)); ax.set_title(f'{c:.2f}'); ax.axis('off')
    plt.tight_layout()

## 5. Latency: đo thay vì đoán

Đường đi thật: `camera/decode → resize + copy GPU → YOLO → NMS → crop/rectify → OCR → rule/DB → barie`. Đo batch=1, warm-up, đồng bộ CUDA và báo median + p95. FPS từ batch lớn là throughput, không phải latency của một xe. YOLO26 có head end-to-end `nms=False`; hãy so cả tốc độ lẫn accuracy với head mặc định. Cell này dùng frame đã decode nên chưa gồm RTSP, OCR và DB.

In [ ]:
frame=cv2.imread(str(test_images[0]))
def sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()
def benchmark(nms,warmup=10,repeats=50):
    kw=dict(imgsz=IMG_SIZE,conf=.25,iou=.70,device=DEVICE,nms=nms,verbose=False)
    for _ in range(warmup): best.predict(frame,**kw)
    sync(); ms=[]; result=None
    for _ in range(repeats):
        sync(); t=time.perf_counter(); result=best.predict(frame,**kw); sync(); ms.append((time.perf_counter()-t)*1000)
    med=statistics.median(ms)
    return {'nms':nms,'median_ms':med,'p95_ms':float(np.percentile(ms,95)),'approx_fps':1000/med,'breakdown':result[0].speed}
for mode in (True,False): print(benchmark(mode))

### Lưu ý production

- Biển quá ít pixel: camera/tiêu cự/ROI thường quan trọng hơn model lớn; tăng input không tạo lại chi tiết đã mất.
- Buffer RTSP có thể gây trễ hàng trăm ms: dùng queue ngắn, bỏ frame cũ và xử lý frame mới nhất.
- Một xe có nhiều frame: tracker/debounce, lấy 3–5 crop tốt và OCR voting; không mở barie mỗi frame.
- Validation phải giống domain thật: ngày/đêm, mưa, đèn pha, xe máy, biển 1/2 dòng. Chia theo camera/video/ngày, không chia ngẫu nhiên các frame cùng clip.
- KPI cuối là exact-match biển, false-open/false-deny và p95 từ ROI đến quyết định; mAP detector chưa phải accuracy toàn hệ thống.
- Biển số là dữ liệu nhạy cảm: giới hạn retention, phân quyền và log truy cập.
- Trước khi đóng gói sản phẩm thương mại, kiểm tra giấy phép Ultralytics (AGPL-3.0/Enterprise) và license CC BY 4.0 của dataset.

Tài liệu chính thức: [YOLO26](https://docs.ultralytics.com/models/yolo26/) · [Train](https://docs.ultralytics.com/modes/train/) · [Validation](https://docs.ultralytics.com/modes/val/) · [Export](https://docs.ultralytics.com/modes/export/)

## 6. Export

ONNX dễ mang sang nhiều môi trường. TensorRT thường nhanh hơn trên NVIDIA nhưng engine phụ thuộc GPU/CUDA/TensorRT, nên build trên máy đích. Sau export phải benchmark lại; latency PyTorch không đại diện file deploy.

In [ ]:
onnx_path=best.export(format='onnx',imgsz=IMG_SIZE,dynamic=False,simplify=True,opset=17,nms=False,device=DEVICE)
print('ONNX:',onnx_path)
EXPORT_TENSORRT=False
if EXPORT_TENSORRT:
    assert torch.cuda.is_available()
    engine=best.export(format='engine',imgsz=IMG_SIZE,batch=1,half=True,dynamic=False,nms=False,device=0,workspace=4)
    print('TensorRT:',engine)
if IN_COLAB:
    from google.colab import files
    files.download(str(BEST_PT))

## Checklist bàn giao

- Lưu `best.pt`, data yaml, version Ultralytics, config và `results.png`.
- Báo precision/recall/mAP50–95 riêng `plate`; test chỉ một lần.
- Báo median/p95 batch=1 trên phần cứng đích, gồm camera + OCR + DB.
- Chỉ nâng `n → s` hoặc `640 → 960` khi số liệu chứng minh recall tăng đủ bù latency.
- Để đọc ký tự, cần ground-truth text và đo exact-match/CER; mAP detection không chứng minh OCR đọc đúng.